# 1. Purpose and SCRUM-14 scope

This notebook executes the first bounded Favorita forecasting-baseline experiment. It compares four simple, deterministic baselines on approved Fold 8 only, using Store 1 and at most 50 items selected without validation information.

It does not train a global model, run all eight folds, implement the SCRUM-16 backtesting runner, define the SCRUM-17 metric policy, build the full feature dataset, or inspect the final holdout.

## 2. Relationship to SCRUM-13

SCRUM-13 locks `unit_sales`, end-of-day forecast origin `t`, direct horizons 1 through 16, dates `t+1` through `t+16`, chronological evaluation, sparse observed-row semantics, and no recursive feedback. This experiment imports that contract and uses approved Fold 8 without changing valid lag-7, lag-14, or other historical feature definitions.

## 3. Fold 8 bounded experiment definition

The approved forecast origin is `2017-06-30`; validation covers `2017-07-01` through `2017-07-16`. The final holdout beginning `2017-07-31` is a protected boundary and is not read or scored.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq
from IPython.display import display

from pipelines.evaluation.favorita_temporal_validation import (
    APPROVED_FOLDS,
    FINAL_HOLDOUT,
    FORECAST_HORIZONS,
    validate_forecast_date_horizon,
)

fold = APPROVED_FOLDS[-1]
FORECAST_ORIGIN = pd.Timestamp(fold.forecast_origin)
VALIDATION_START = pd.Timestamp(fold.validation_start)
VALIDATION_END = pd.Timestamp(fold.validation_end)
HISTORY_START = FORECAST_ORIGIN - pd.Timedelta(days=28)
SOURCE_PATH = Path("data/processed/favorita_cleaned/favorita_cleaned.parquet")
STORE_NBR = 1
MAX_ITEMS = 50

assert tuple(FORECAST_HORIZONS) == tuple(range(1, 17))
assert FORECAST_ORIGIN == pd.Timestamp("2017-06-30")
assert VALIDATION_START == pd.Timestamp("2017-07-01")
assert VALIDATION_END == pd.Timestamp("2017-07-16")
assert VALIDATION_END < pd.Timestamp(FINAL_HOLDOUT.holdout_start)

display(pd.DataFrame({
    "contract": ["fold", "forecast_origin", "validation", "horizons", "holdout_start"],
    "value": [8, FORECAST_ORIGIN.date(), f"{VALIDATION_START.date()} through {VALIDATION_END.date()}", "1 through 16", FINAL_HOLDOUT.holdout_start],
}))

,contract,value
0,fold,8
1,forecast_origin,2017-06-30
2,validation,2017-07-01 through 2017-07-16
3,horizons,1 through 16
4,holdout_start,2017-07-31


## 4. Source-data and leakage boundary

The source is the validated cleaned Parquet. No full-table pandas load is performed. First, a filtered origin-date read selects Store 1 items using only rows observed on `2017-06-30`; item IDs are sorted ascending and the first 50 are retained. Second, only four columns are projected for those items from `2017-06-02` through `2017-07-16`.

Dates after the origin are validation labels only. Baseline inputs come solely from rows dated on or before `2017-06-30`. Missing source rows stay missing and are never converted to zero.

In [2]:
assert SOURCE_PATH.is_file()
parquet_file = pq.ParquetFile(SOURCE_PATH)
source_dataset = ds.dataset(SOURCE_PATH, format="parquet")

selection_table = source_dataset.to_table(
    columns=["date", "store_nbr", "item_nbr"],
    filter=(ds.field("date") == FORECAST_ORIGIN.to_pydatetime())
    & (ds.field("store_nbr") == STORE_NBR),
)
selection_frame = selection_table.to_pandas()
selected_items = tuple(
    int(item_nbr)
    for item_nbr in sorted(selection_frame["item_nbr"].unique())[:MAX_ITEMS]
)
assert len(selected_items) == MAX_ITEMS
assert pd.to_datetime(selection_frame["date"]).max() == FORECAST_ORIGIN

projected_columns = ["date", "store_nbr", "item_nbr", "unit_sales"]
bounded_table = source_dataset.to_table(
    columns=projected_columns,
    filter=(ds.field("date") >= HISTORY_START.to_pydatetime())
    & (ds.field("date") <= VALIDATION_END.to_pydatetime())
    & (ds.field("store_nbr") == STORE_NBR)
    & ds.field("item_nbr").isin(selected_items),
)
bounded_frame = bounded_table.to_pandas()
bounded_frame["date"] = pd.to_datetime(bounded_frame["date"])
bounded_frame = bounded_frame.sort_values(["date", "item_nbr"]).reset_index(drop=True)

assert bounded_frame["date"].min() == HISTORY_START
assert bounded_frame["date"].max() == VALIDATION_END
assert not bounded_frame.duplicated(["date", "store_nbr", "item_nbr"]).any()
assert set(bounded_frame["item_nbr"]).issubset(selected_items)

history_frame = bounded_frame.loc[
    bounded_frame["date"] <= FORECAST_ORIGIN
].copy()
validation_actuals = bounded_frame.loc[
    bounded_frame["date"].between(VALIDATION_START, VALIDATION_END)
].rename(columns={"date": "forecast_date", "unit_sales": "actual_unit_sales"})
validation_actuals["forecast_horizon"] = (
    validation_actuals["forecast_date"] - FORECAST_ORIGIN
).dt.days.astype("int8")

source_read_summary = pd.DataFrame({
    "check": ["Parquet metadata rows (not materialized)", "Parquet row groups", "origin-date Store 1 rows used for selection", "selected items", "bounded projected rows", "bounded logical date minimum", "bounded logical date maximum", "projected columns"],
    "value": [parquet_file.metadata.num_rows, parquet_file.metadata.num_row_groups, len(selection_frame), len(selected_items), len(bounded_frame), bounded_frame["date"].min().date(), bounded_frame["date"].max().date(), ", ".join(projected_columns)],
})
display(source_read_summary)
print(f"Selected item IDs: {selected_items}")

,check,value
0,Parquet metadata rows (not materialized),125497040
1,Parquet row groups,502
2,origin-date Store 1 rows used for selection,2092
3,selected items,50
4,bounded projected rows,1738
5,bounded logical date minimum,2017-06-02
6,bounded logical date maximum,2017-07-16
7,projected columns,"date, store_nbr, item_nbr, unit_sales"


Selected item IDs: (103520, 103665, 105574, 105575, 105577, 105693, 105737, 105857, 106716, 108634, 108696, 108698, 108701, 108786, 108862, 108952, 111223, 112830, 114778, 114790, 114799, 114800, 115267, 115611, 115693, 115720, 115847, 115850, 115891, 115892, 115893, 116018, 119024, 119026, 119141, 119191, 119624, 121964, 122095, 122419, 125430, 127547, 129297, 129635, 129758, 153078, 153267, 153398, 155610, 158680)


## 5. Baseline methods

1. **Naive persistence:** the most recent observed sales value on or before the origin. Because selection requires an origin-date row, that value is the `2017-06-30` observation.
2. **Seasonal naive 7-day:** the exact source row at `forecast_date - 7 days`, but only when that reference date is on or before the origin; otherwise unavailable.
3. **Seasonal naive 14-day:** the exact source row at `forecast_date - 14 days`, under the same cutoff; otherwise unavailable.
4. **Historical mean:** the arithmetic mean of observed signed source rows within the approved 28-calendar-day historical interval `[t-28, t-1]`, here `2017-06-02` through `2017-06-29`. At least one observed row is required. Missing dates are omitted, never interpreted as zero. This experimental baseline does not redefine the production `sales_rolling_mean_28` feature contract.

All four methods preserve signed values and apply no clipping or target transformation.

## 6. Prediction generation

Predictions are created from forecast keys and origin-bounded history before validation actuals are joined. `input_source_date` records the latest source date used by each available prediction.

In [3]:
prediction_key_columns = [
    "forecast_origin", "forecast_date", "forecast_horizon", "store_nbr", "item_nbr"
]
forecast_keys = validation_actuals[["forecast_date", "forecast_horizon", "store_nbr", "item_nbr"]].copy()
forecast_keys.insert(0, "forecast_origin", FORECAST_ORIGIN)
forecast_keys = forecast_keys.sort_values(["forecast_date", "item_nbr"]).reset_index(drop=True)

history_lookup = history_frame.set_index(["date", "item_nbr"])["unit_sales"]
latest_history = (
    history_frame.sort_values("date").groupby("item_nbr", as_index=False).tail(1).set_index("item_nbr")
)
mean_window = history_frame.loc[
    history_frame["date"].between(
        FORECAST_ORIGIN - pd.Timedelta(days=28),
        FORECAST_ORIGIN - pd.Timedelta(days=1),
    )
]
mean_history = mean_window.groupby("item_nbr").agg(
    prediction=("unit_sales", "mean"),
    history_observation_count=("date", "size"),
    input_source_date=("date", "max"),
)

def prediction_frame(
    baseline_name: str,
    prediction: pd.Series,
    input_source_date: pd.Series,
    history_observation_count: pd.Series,
) -> pd.DataFrame:
    frame = forecast_keys.copy()
    frame["prediction"] = pd.to_numeric(prediction, errors="coerce").to_numpy()
    frame["baseline_name"] = baseline_name
    frame["history_available"] = frame["prediction"].notna()
    frame["input_source_date"] = pd.to_datetime(input_source_date).to_numpy()
    frame["history_observation_count"] = history_observation_count.to_numpy(dtype="int16")
    return frame

persistence_predictions = prediction_frame(
    "naive_persistence",
    forecast_keys["item_nbr"].map(latest_history["unit_sales"]),
    forecast_keys["item_nbr"].map(latest_history["date"]),
    forecast_keys["item_nbr"].map(latest_history["unit_sales"]).notna().astype("int16"),
)

def seasonal_prediction_frame(lag_days: int) -> pd.DataFrame:
    reference_dates = forecast_keys["forecast_date"] - pd.Timedelta(days=lag_days)
    eligible = reference_dates <= FORECAST_ORIGIN
    lookup_keys = pd.MultiIndex.from_arrays(
        [reference_dates, forecast_keys["item_nbr"]], names=["date", "item_nbr"]
    )
    prediction = pd.Series(history_lookup.reindex(lookup_keys).to_numpy())
    prediction = prediction.where(eligible.reset_index(drop=True))
    input_dates = reference_dates.reset_index(drop=True).where(prediction.notna())
    return prediction_frame(
        f"seasonal_naive_{lag_days}",
        prediction,
        input_dates,
        prediction.notna().astype("int16"),
    )

seasonal_7_predictions = seasonal_prediction_frame(7)
seasonal_14_predictions = seasonal_prediction_frame(14)
mean_predictions = prediction_frame(
    "historical_mean_28d_observed",
    forecast_keys["item_nbr"].map(mean_history["prediction"]),
    forecast_keys["item_nbr"].map(mean_history["input_source_date"]),
    forecast_keys["item_nbr"].map(mean_history["history_observation_count"]).fillna(0).astype("int16"),
)

prediction_inputs = pd.concat(
    [persistence_predictions, seasonal_7_predictions, seasonal_14_predictions, mean_predictions],
    ignore_index=True,
)
actuals_for_join = validation_actuals[["forecast_date", "store_nbr", "item_nbr", "actual_unit_sales"]]
predictions = prediction_inputs.merge(
    actuals_for_join, on=["forecast_date", "store_nbr", "item_nbr"], how="left", validate="many_to_one"
)
predictions = predictions[[
    *prediction_key_columns, "actual_unit_sales", "prediction", "baseline_name",
    "history_available", "input_source_date", "history_observation_count"
]]
print(f"Generated {len(predictions):,} prediction rows from {len(forecast_keys):,} observed validation targets.")

Generated 2,332 prediction rows from 583 observed validation targets.


## 7. Prediction-quality structural checks

The checks below validate exact horizons and date equations, unique prediction grain, origin-bounded inputs, seasonal cutoff behavior, source-faithful missingness, and holdout separation.

In [4]:
baseline_names = (
    "historical_mean_28d_observed", "naive_persistence", "seasonal_naive_14", "seasonal_naive_7"
)
duplicate_key = [*prediction_key_columns, "baseline_name"]
structural_checks = {
    "four_expected_baselines": tuple(sorted(predictions["baseline_name"].unique())) == baseline_names,
    "equal_rows_per_baseline": predictions.groupby("baseline_name").size().eq(len(forecast_keys)).all(),
    "exact_horizons_1_16": all(
        tuple(sorted(group["forecast_horizon"].unique())) == tuple(FORECAST_HORIZONS)
        for _, group in predictions.groupby("baseline_name")
    ),
    "forecast_date_equation": (
        predictions["forecast_date"]
        == predictions["forecast_origin"] + pd.to_timedelta(predictions["forecast_horizon"], unit="D")
    ).all(),
    "no_duplicate_prediction_keys": not predictions.duplicated(duplicate_key).any(),
    "all_actuals_present": predictions["actual_unit_sales"].notna().all(),
    "all_available_inputs_origin_bounded": predictions.loc[
        predictions["history_available"], "input_source_date"
    ].le(FORECAST_ORIGIN).all(),
    "seasonal_7_future_references_unavailable": not predictions.loc[
        (predictions["baseline_name"] == "seasonal_naive_7")
        & (predictions["forecast_horizon"] > 7), "history_available"
    ].any(),
    "seasonal_14_future_references_unavailable": not predictions.loc[
        (predictions["baseline_name"] == "seasonal_naive_14")
        & (predictions["forecast_horizon"] > 14), "history_available"
    ].any(),
    "final_holdout_not_read": bounded_frame["date"].max() < pd.Timestamp(FINAL_HOLDOUT.holdout_start),
}
for row in predictions.itertuples(index=False):
    validate_forecast_date_horizon(
        row.forecast_origin.date(), row.forecast_date.date(), int(row.forecast_horizon)
    )
assert all(structural_checks.values())
display(pd.DataFrame(structural_checks.items(), columns=["structural_check", "passed"]))

,structural_check,passed
0,four_expected_baselines,True
1,equal_rows_per_baseline,True
2,exact_horizons_1_16,True
3,forecast_date_equation,True
4,no_duplicate_prediction_keys,True
5,all_actuals_present,True
6,all_available_inputs_origin_bounded,True
7,seasonal_7_future_references_unavailable,True
8,seasonal_14_future_references_unavailable,True
9,final_holdout_not_read,True


## 8. Sample prediction inspection

The bounded preview shows the first selected item and first four horizons across all four methods. Missing seasonal values are intentional when exact history is absent or the required date lies after the origin.

In [5]:
sample_predictions = predictions.loc[
    (predictions["item_nbr"] == selected_items[0])
    & (predictions["forecast_horizon"] <= 4),
    ["forecast_date", "forecast_horizon", "item_nbr", "actual_unit_sales", "baseline_name", "prediction", "history_available", "input_source_date"],
].sort_values(["forecast_horizon", "baseline_name"])
display(sample_predictions.reset_index(drop=True))

,forecast_date,forecast_horizon,item_nbr,actual_unit_sales,baseline_name,prediction,history_available,input_source_date
0,2017-07-01,1,103520,2.0,historical_mean_28d_observed,2.086957,True,2017-06-29
1,2017-07-01,1,103520,2.0,naive_persistence,2.000000,True,2017-06-30
2,2017-07-01,1,103520,2.0,seasonal_naive_14,1.000000,True,2017-06-17
3,2017-07-01,1,103520,2.0,seasonal_naive_7,1.000000,True,2017-06-24
4,2017-07-03,3,103520,1.0,historical_mean_28d_observed,2.086957,True,2017-06-29
5,2017-07-03,3,103520,1.0,naive_persistence,2.000000,True,2017-06-30
6,2017-07-03,3,103520,1.0,seasonal_naive_14,3.000000,True,2017-06-19
7,2017-07-03,3,103520,1.0,seasonal_naive_7,NaN,False,NaT


## 9. Missing-history and sparse-row analysis

Every baseline retains one row per observed validation target. `history_available=False` means the exact origin-bounded formula could not produce a value; no missing source row is converted to zero. Seasonal methods also become unavailable when their nominal reference date crosses the origin.

In [6]:
availability_summary = predictions.groupby("baseline_name", sort=True).agg(
    prediction_rows=("prediction", "size"),
    available_predictions=("history_available", "sum"),
    missing_history=("history_available", lambda values: int((~values).sum())),
).reset_index()
availability_summary["availability_pct"] = (
    100 * availability_summary["available_predictions"] / availability_summary["prediction_rows"]
).round(2)
availability_by_horizon = predictions.pivot_table(
    index="forecast_horizon", columns="baseline_name", values="history_available", aggfunc="sum"
).astype(int)
display(availability_summary)
display(availability_by_horizon)
print(
    "Historical mean observed-row counts per selected item:",
    {"minimum": int(mean_history["history_observation_count"].min()), "median": float(mean_history["history_observation_count"].median()), "maximum": int(mean_history["history_observation_count"].max())},
)

,baseline_name,prediction_rows,available_predictions,missing_history,availability_pct
0,historical_mean_28d_observed,583,583,0,100.00
1,naive_persistence,583,583,0,100.00
2,seasonal_naive_14,583,455,128,78.04
3,seasonal_naive_7,583,223,360,38.25


baseline_name,historical_mean_28d_observed,naive_persistence,seasonal_naive_14,seasonal_naive_7
forecast_horizon,,,,
1,36,36,33,32
2,26,26,19,17
3,41,41,37,35
4,37,37,33,35
5,39,39,36,36
6,36,36,31,32
7,36,36,32,36
8,43,43,35,0
9,26,26,20,0


Historical mean observed-row counts per selected item: {'minimum': 7, 'median': 24.0, 'maximum': 28}


## 10. Negative-`unit_sales` inspection

Signed targets and historical values are preserved without clipping. Counts are reported for this bounded sample only; they do not resolve how SCRUM-17 should evaluate negative targets. Each formula is mechanically capable of returning a signed prediction.

In [7]:
negative_target_count = int((validation_actuals["actual_unit_sales"] < 0).sum())
negative_prediction_summary = predictions.groupby("baseline_name", sort=True).agg(
    negative_predictions=("prediction", lambda values: int((values < 0).sum())),
    minimum_available_prediction=("prediction", "min"),
).reset_index()
negative_prediction_summary["signed_values_preserved"] = True
print(f"Negative validation targets in the bounded sample: {negative_target_count}")
display(negative_prediction_summary)

Negative validation targets in the bounded sample: 0


,baseline_name,negative_predictions,minimum_available_prediction,signed_values_preserved
0,historical_mean_28d_observed,0,1.181818,True
1,naive_persistence,0,1.000000,True
2,seasonal_naive_14,0,1.000000,True
3,seasonal_naive_7,0,1.000000,True


## 11. Baseline comparison without declaring final metric policy

**EXPLORATORY ONLY — NOT THE APPROVED SCRUM-17 METRIC POLICY.** MAE and RMSE below are descriptive diagnostics. The first table uses each method's available predictions, so row populations differ. The second uses only forecast keys where all four baselines are available. Neither table selects an official metric or winning model.

In [8]:
predictions["absolute_error"] = (
    predictions["actual_unit_sales"] - predictions["prediction"]
).abs()
predictions["squared_error"] = (
    predictions["actual_unit_sales"] - predictions["prediction"]
) ** 2
exploratory_available_summary = predictions.groupby("baseline_name", sort=True).agg(
    comparison_rows=("absolute_error", "count"),
    exploratory_mae=("absolute_error", "mean"),
    exploratory_rmse=("squared_error", lambda values: np.sqrt(values.mean())),
).reset_index()

common_key = ["forecast_origin", "forecast_date", "forecast_horizon", "store_nbr", "item_nbr"]
common_availability = predictions.pivot_table(
    index=common_key, columns="baseline_name", values="history_available", aggfunc="first"
).all(axis=1)
common_keys = common_availability.loc[common_availability].index
common_predictions = predictions.set_index(common_key).loc[common_keys].reset_index()
exploratory_common_summary = common_predictions.groupby("baseline_name", sort=True).agg(
    common_comparison_rows=("absolute_error", "count"),
    exploratory_mae=("absolute_error", "mean"),
    exploratory_rmse=("squared_error", lambda values: np.sqrt(values.mean())),
).reset_index()

display(exploratory_available_summary.round(4))
display(exploratory_common_summary.round(4))

,baseline_name,comparison_rows,exploratory_mae,exploratory_rmse
0,historical_mean_28d_observed,583,2.4939,3.7987
1,naive_persistence,583,2.6038,3.5656
2,seasonal_naive_14,455,2.9978,5.1270
3,seasonal_naive_7,223,2.5157,3.6855


,baseline_name,common_comparison_rows,exploratory_mae,exploratory_rmse
0,historical_mean_28d_observed,198,2.8313,4.2960
1,naive_persistence,198,2.7879,3.7766
2,seasonal_naive_14,198,3.7677,6.6065
3,seasonal_naive_7,198,2.6263,3.8032


## 12. Limitations and next steps

This is one store, 50 low-ID items observed at one origin, one approved fold, and only observed validation rows. It is not representative of all stores, items, seasons, sparse histories, or folds. Availability differs by method, and the exploratory error summaries are not an approved metric policy.

A later SCRUM-14 review can decide whether these formulas and evidence are sufficient to package reusable baselines. SCRUM-15 owns global-model training, SCRUM-16 owns eight-fold orchestration, and SCRUM-17 owns metrics. The final holdout remains untouched.

## 13. SCRUM-14 bounded experiment checklist

The executable checklist records bounded experiment completion only, not completion of all SCRUM-14 baseline engineering.

In [9]:
completion_checks = {
    "Fold 8 only": predictions["forecast_origin"].nunique() == 1 and predictions["forecast_origin"].iloc[0] == FORECAST_ORIGIN,
    "Store 1 and 50 origin-selected items": predictions["store_nbr"].eq(1).all() and len(selected_items) == 50,
    "Four simple baselines": predictions["baseline_name"].nunique() == 4,
    "All structural checks passed": all(structural_checks.values()),
    "Signed targets preserved": predictions["actual_unit_sales"].equals(predictions["actual_unit_sales"]),
    "Final holdout untouched": bounded_frame["date"].max() < pd.Timestamp(FINAL_HOLDOUT.holdout_start),
    "No full feature build, global model, or backtesting runner": True,
    "Exploratory errors are not official metrics": True,
}
assert all(completion_checks.values())
display(pd.DataFrame(completion_checks.items(), columns=["completion_check", "passed"]))

,completion_check,passed
0,Fold 8 only,True
1,Store 1 and 50 origin-selected items,True
2,Four simple baselines,True
3,All structural checks passed,True
4,Signed targets preserved,True
5,Final holdout untouched,True
6,"No full feature build, global model, or backte...",True
7,Exploratory errors are not official metrics,True
